# Architecture Recovery for the Zeeguu API Application

This notebook is my attempt at a static architecture recovery of the Zeeguu API by parsing Python imports with `ast`, building dependency graphs with `networkx`, and exporting interactive visualizations with `pyvis`.

An edge `A → B` means that module or package `A` imports `B`.

## 1. Dependency View
In the following sections we make code to create a Dependency view of the Zeeguu API. 
Creating graphs to show which packages import which packages and different depths. 

### 1.1 Imports and Configuration

Set `CODE_ROOT_FOLDER` to the local path of the Zeeguu API repository you want to analyse.

In [320]:
import os
import ast
import warnings
from pathlib import Path

import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display

# Suppress SyntaxWarning noise from parsed files
warnings.filterwarnings("ignore", category=SyntaxWarning)

# Print current working directory if needed
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# Set path to the repository you are doing architecture recovery on.
CODE_ROOT_FOLDER = "../zeeguu-api/"
print(f"Code root folder: {CODE_ROOT_FOLDER}")

Current working directory: /mnt/d/GitHub/software-architecture-individual
Code root folder: ../zeeguu-api/


### 1.2 Path and Module Helper Functions

In [321]:
def file_path(file_name: str) -> str:
  """
  Get the full path of a file in the codebase from its relative path.
  """
  return CODE_ROOT_FOLDER + file_name


def module_name_from_file_path(full_path: str) -> str:
  """
  Convert a Python file path to a Python module name.

  Example:
      ../zeeguu-api/zeeguu/core/model/user.py
      -> zeeguu.core.model.user
  """
  file_name = full_path[len(CODE_ROOT_FOLDER):]
  file_name = file_name.replace("/__init__.py", "")
  file_name = file_name.replace("/", ".")
  file_name = file_name.replace(".py", "")
  return file_name


def in_package(module_name: str, package_prefix: str) -> bool:
  """
  Check whether a module belongs to a package.
  """
  return module_name == package_prefix or module_name.startswith(package_prefix + ".")


def package_of(module_name: str, depth: int = 3) -> str:
  """
  Convert a module name to a package name at a given depth.
  Depth set to 3 as default.

  Example:
      zeeguu.core.model.user -> zeeguu.core.model, with depth=3
  """
  return ".".join(module_name.split(".")[:depth])

### 1.3 Sanity Checks

These assertions verify that the path/module conversion helpers behave as expected.

In [322]:
assert file_path("zeeguu/core/model/user.py") == "../zeeguu-api/zeeguu/core/model/user.py"
assert module_name_from_file_path(file_path("zeeguu/core/model/user.py")) == "zeeguu.core.model.user"

print("Helper function sanity checks passed.")

Helper function sanity checks passed.


### 1.4 Extract Imports from Python Files

This function uses the Python `ast` module to parse imports without executing the target files.

In [323]:
def imports_from_file(file_path: str) -> list[str]:
  """
  Extract imported modules from a Python file using ast.

  Handles:
  - import x
  - import x, y
  - from x import y
  - from . import y, represented as "."
  """
  imports = []

  with open(file_path, encoding="utf-8", errors="ignore") as f:
    tree = ast.parse(f.read(), filename=file_path)

  for node in ast.walk(tree):
    # import x, y, z
    if isinstance(node, ast.Import):
      for name in node.names:
        imports.append(name.name)

    # from x import y, z
    elif isinstance(node, ast.ImportFrom):
      if node.module is not None:
        imports.append(node.module)
      else:
        # handles: from . import y
        imports.append(".")

  return imports

### 1.5 Inspect Imports from Example Files

As a sanity check I list the imports from three files from the model in Zeeguu API, namely: user.py, bookmark.py, and unique_code.py

In [324]:
user_imports = imports_from_file(file_path("zeeguu/core/model/user.py"))
bookmark_imports = imports_from_file(file_path("zeeguu/core/model/bookmark.py"))
unique_code_imports = imports_from_file(file_path("zeeguu/core/model/unique_code.py"))

print("Imports from user.py:")
print(user_imports)

print("\nImports from bookmark.py:")
print(bookmark_imports)

print("\nImports from unique_code.py:")
print(unique_code_imports)

assert unique_code_imports != bookmark_imports
print("\nImport extraction sanity check passed.")

Imports from user.py:
['datetime', 'json', 'random', 're', 'sqlalchemy.orm', 'sqlalchemy', 'sqlalchemy.orm', 'sqlalchemy.orm.exc', 'zeeguu.core', 'zeeguu.core.language.difficulty_estimator_factory', 'zeeguu.core.model.db', 'zeeguu.core.model.language', 'zeeguu.core.util.hash', 'zeeguu.logging', 'zeeguu.logging', 'datetime', 'zeeguu.core.model.user_cohort_map', 'datetime', 'zeeguu.core.model', 'zeeguu.core.model.bookmark', 'zeeguu.core.model.user_avatar', 'zeeguu.core.model.user_word', 'zeeguu.core.model.user_preference', 'zeeguu.core.model', 'zeeguu.core.model', 'zeeguu.core.model.user_language', 'zeeguu.core.word_scheduling.basicSR.basicSR', 'zeeguu.core.sql.queries.query_loader', 'zeeguu.core.sql.query_building', 'zeeguu.core.model.bookmark', 'zeeguu.core.model.user_word', 'zeeguu.core.model.exercise', 'zeeguu.core.model.phrase', 'zeeguu.core.model.meaning', 'zeeguu.core.model.user_word', 'zeeguu.core.model.exercise', 'zeeguu.core.model.phrase', 'zeeguu.core.model.meaning', 'zeeguu.c

### 1.6 Graph Drawing

The graph is saved as an interactive HTML file under `./graphs/`. In a notebook, the generated HTML is also displayed inline using an `IFrame`.

In [325]:
def draw_graph(G: nx.DiGraph, figure_name: str, height: str = "900px", width: str = "100%", notebook: bool = True):
  """
  Draw a directed graph using PyVis and save it as an interactive HTML file.

  Edge A -> B means A imports B.
  """
  os.makedirs("./graphs", exist_ok=True)

  net = Network(
    height=height,
    width=width,
    directed=True,
    notebook=notebook,
    bgcolor="#ffffff",
    font_color="#000000",
  )

  # Optional physics layout settings
  net.barnes_hut(
    gravity=-30000,
    central_gravity=0.3,
    spring_length=200,
    spring_strength=0.05,
    damping=0.09,
    overlap=0,
  )

  # Add nodes
  for node in G.nodes:
    net.add_node(node, label=node, title=node)

  # Add edges
  for source, target in G.edges:
    net.add_edge(source, target, title=f"{source} imports {target}")

  # Add interaction buttons
  net.show_buttons(filter_=["physics"])

  output_path = f"./graphs/{figure_name}.html"
  net.write_html(output_path)

  print(f"Saved interactive graph to: {output_path}")
  # display(IFrame(src=output_path, width="100%", height=height.replace("px", "")))

  return output_path

### 1.7 Build Module-Level Dependency Graph

This graph keeps dependencies at full module granularity.

In [326]:
def dependencies_digraph(code_root_folder: str) -> nx.DiGraph:
  """
  Build a directed graph of module dependencies.

  Edge A -> B means module A imports module B.
  """
  files = Path(code_root_folder).rglob("*.py")
  G = nx.DiGraph()

  for file in files:
    source_file_path = str(file)
    source_module = module_name_from_file_path(source_file_path)

    if source_module not in G.nodes:
      G.add_node(source_module)

    for target_module in imports_from_file(source_file_path):
      if target_module.startswith("zeeguu"):
        G.add_edge(source_module, target_module)

  return G

### 1.8 Build Package-Level Dependency Graph

This graph groups modules into packages according to the chosen depth.

Examples:

- `depth=2`: `zeeguu.core`
- `depth=3`: `zeeguu.core.model`
- `depth=4`: `zeeguu.core.model.user`

In [327]:
def package_dependencies_digraph(code_root_folder: str, depth: int = 3) -> nx.DiGraph:
  """
  Build a directed graph of package dependencies.

  Example:
    zeeguu.core.model.user imports zeeguu.core.model.bookmark
    becomes zeeguu.core.model -> zeeguu.core.model when depth=3.

  Self-dependencies are omitted.
  """
  files = Path(code_root_folder).rglob("*.py")
  G = nx.DiGraph()

  for file in files:
    source_module = module_name_from_file_path(str(file))

    if not source_module.startswith("zeeguu"):
      continue

    source_pkg = package_of(source_module, depth)
    G.add_node(source_pkg)

    for target_module in imports_from_file(str(file)):
      if target_module.startswith("zeeguu"):
        target_pkg = package_of(target_module, depth)
        if source_pkg != target_pkg:
          G.add_edge(source_pkg, target_pkg)

  return G

### 1.9 Generate Architecture Graphs at Different Granularities

In [328]:
DG_3 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=3)
print(f"Depth 3 graph: {DG_3.number_of_nodes()} nodes, {DG_3.number_of_edges()} edges")
draw_graph(DG_3, "architecture-packages")

Depth 3 graph: 59 nodes, 192 edges
Saved interactive graph to: ./graphs/architecture-packages.html


'./graphs/architecture-packages.html'

In [329]:
DG_2 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=2)
print(f"Depth 2 graph: {DG_2.number_of_nodes()} nodes, {DG_2.number_of_edges()} edges")
draw_graph(DG_2, "architecture-packages-depth-2")

Depth 2 graph: 8 nodes, 13 edges
Saved interactive graph to: ./graphs/architecture-packages-depth-2.html


'./graphs/architecture-packages-depth-2.html'

In [330]:
DG_4 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=4)
print(f"Depth 4 graph: {DG_4.number_of_nodes()} nodes, {DG_4.number_of_edges()} edges")
draw_graph(DG_4, "architecture-packages-depth-4")

Depth 4 graph: 370 nodes, 1223 edges
Saved interactive graph to: ./graphs/architecture-packages-depth-4.html


'./graphs/architecture-packages-depth-4.html'

### 1.10 Subgraphs of Specific Packages

In [331]:
# Core package: main domain logic, including models, services, and repositories.
DG_core = DG_3.subgraph([n for n in DG_3.nodes if n.startswith("zeeguu.core")]).copy()
print(f"Core subgraph: {DG_core.number_of_nodes()} nodes, {DG_core.number_of_edges()} edges")
draw_graph(DG_core, "architecture-packages-core")

Core subgraph: 42 nodes, 110 edges
Saved interactive graph to: ./graphs/architecture-packages-core.html


'./graphs/architecture-packages-core.html'

In [332]:
DG_model = DG_4.subgraph([n for n in DG_4.nodes if n.startswith("zeeguu.core.model")]).copy()
print(f"Model subgraph: {DG_model.number_of_nodes()} nodes, {DG_model.number_of_edges()} edges")
draw_graph(DG_model, "architecture-packages-core-model")

Model subgraph: 104 nodes, 334 edges
Saved interactive graph to: ./graphs/architecture-packages-core-model.html


'./graphs/architecture-packages-core-model.html'

### 1.11 Basic Graph Metrics

These metrics can help you identify central packages/modules and possible architectural hotspots.

In [333]:
def print_graph_summary(G: nx.DiGraph, name: str):
  print(f"{name}")
  print("-" * len(name))
  print(f"Nodes: {G.number_of_nodes()}")
  print(f"Edges: {G.number_of_edges()}")

  if G.number_of_nodes() > 0:
    in_degree = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:10]
    out_degree = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:10]

    print("\nTop imported packages/modules:")
    for node, degree in in_degree:
      print(f"  {node}: {degree}")

    print("\nTop importing packages/modules:")
    for node, degree in out_degree:
      print(f"  {node}: {degree}")

print_graph_summary(DG_3, "Package Dependency Graph, Depth 3")

Package Dependency Graph, Depth 3
---------------------------------
Nodes: 59
Edges: 192

Top imported packages/modules:
  zeeguu.core.model: 34
  zeeguu.logging: 19
  zeeguu.core: 14
  zeeguu: 8
  zeeguu.core.util: 8
  zeeguu.core.tokenization: 7
  zeeguu.core.word_scheduling: 7
  zeeguu.core.llm_services: 6
  zeeguu.config: 5
  zeeguu.core.constants: 5

Top importing packages/modules:
  zeeguu.api.endpoints: 30
  zeeguu.core.model: 23
  zeeguu.core.test: 14
  zeeguu.core.content_retriever: 10
  zeeguu.api.app: 9
  zeeguu.core.audio_lessons: 7
  zeeguu.operations.crawler: 6
  zeeguu.core.llm_services: 6
  zeeguu.core.util: 6
  zeeguu.core.semantic_search: 5


## 2. Domain Model view

Having a dependency graph for the model package can be a good way to get a baseline view at which files import other files, but it is not necessarily equivalent to a Domain model. 

In the following sections make methods to help structure a proper Domain Model View. 

### 2.1 Extract classes from files



In [334]:
def classes_from_file(file_path: str) -> list[str]:
  """
  Extract class names defined in a Python file.
  """
  classes = []

  with open(file_path, encoding="utf-8", errors="ignore") as f:
    tree = ast.parse(f.read(), filename=file_path)

  for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef):
      classes.append(node.name)

  return classes


### 2.2 Find all model classes in zeeguu.core.model

I expect the domain model to be in the zeeguu.core.model package. Therefore I make that the default package to check in my method for finding model_classes.

In [335]:

def model_classes(code_root_folder: str, model_package: str = "zeeguu.core.model") -> dict[str, str]:
  """
  Find classes defined inside the model package.

  Returns:
    Dictionary mapping class name -> module name

  Example:
    {
      "User": "zeeguu.core.model.user",
      "Bookmark": "zeeguu.core.model.bookmark"
    }
  """
  class_to_module = {}

  for file in Path(code_root_folder).rglob("*.py"):
    module_name = module_name_from_file_path(str(file))

    if not module_name.startswith(model_package):
      continue

    for class_name in classes_from_file(str(file)):
      class_to_module[class_name] = module_name

  return class_to_module


### 2.3 Extract class references from a file

With the set of model classes in from the model_classes method we can look for references to these classes in the other model classes.

As an example, if user.py contains: 
- Language
- Article
- Bookmark

Then it's possible to infer that User has a relation to those concepts. 

In [336]:

def class_references_from_file(file_path: str, known_classes: set[str]) -> set[str]:
  """
  Find references to known model classes inside a Python file.

  This detects references such as:
    Bookmark
    Article
    User

  It does not execute the code. It only parses the AST.
  """
  references = set()

  with open(file_path, encoding="utf-8", errors="ignore") as f:
    tree = ast.parse(f.read(), filename=file_path)

  for node in ast.walk(tree):
    if isinstance(node, ast.Name):
      if node.id in known_classes:
        references.add(node.id)

  return references


### 2.4 Build domain model graph

In [337]:
def domain_model_digraph(code_root_folder: str, model_package: str = "zeeguu.core.model") -> nx.DiGraph:
  """
  Build a domain model graph based on model classes and references between them.

  Nodes are model classes.
  Edge A -> B means class A's module references class B somewhere in the code.
  """
  class_to_module = model_classes(code_root_folder, model_package)
  known_classes = set(class_to_module.keys())

  G = nx.DiGraph()

  # Add all model classes as nodes
  for class_name, module_name in class_to_module.items():
    G.add_node(
      class_name,
      module=module_name
    )

  # Search for references between model classes
  for file in Path(code_root_folder).rglob("*.py"):
    module_name = module_name_from_file_path(str(file))

    if not module_name.startswith(model_package):
      continue

    source_classes = classes_from_file(str(file))

    if not source_classes:
      continue

    referenced_classes = class_references_from_file(str(file), known_classes)

    for source_class in source_classes:
      for target_class in referenced_classes:
        if source_class != target_class:
            G.add_edge(source_class, target_class)

  return G


### 2.5 Draw domain model graph

This method isn't really necessary as draw_graph from earlier could be used. draw_graph however labels using module names. This method instead labels nice for a class-level domain model.

In [338]:
def draw_domain_model_graph(G: nx.DiGraph, figure_name: str, height: str = "900px", width: str = "100%", notebook: bool = True):
  """
  Draw a domain model graph using PyVis.

  Edge A -> B means domain class A references domain class B.
  """
  os.makedirs("./graphs", exist_ok=True)

  net = Network(
    height=height,
    width=width,
    directed=True,
    notebook=notebook,
    bgcolor="#ffffff",
    font_color="#000000",
  )

  net.barnes_hut(
    gravity=-30000,
    central_gravity=0.3,
    spring_length=200,
    spring_strength=0.05,
    damping=0.09,
    overlap=0,
  )

  # Add class nodes
  for node, data in G.nodes(data=True):
    module_name = data.get("module", "")

    net.add_node(
      node,
      label=node,
      title=f"{node}<br>{module_name}",
    )

  # Add edges
  for source, target in G.edges:
    net.add_edge(
      source,
      target,
      title=f"{source} references {target}",
    )

  net.show_buttons(filter_=["physics"])

  output_path = f"./graphs/{figure_name}.html"
  net.write_html(output_path)

  print(f"Saved interactive domain model graph to: {output_path}")
  # display(IFrame(src=output_path, width="100%", height=height.replace("px", "")))

  return output_path


### 2.6 Generate the domain model view

In [339]:
DMG = domain_model_digraph(CODE_ROOT_FOLDER)

print(f"Domain model graph: {DMG.number_of_nodes()} classes, {DMG.number_of_edges()} relationships")

draw_domain_model_graph(DMG, "domain-model-view")

Domain model graph: 113 classes, 314 relationships
Saved interactive domain model graph to: ./graphs/domain-model-view.html


'./graphs/domain-model-view.html'

### 2.7 Most connected domain classes

Printing the most connected domain classes help identify central domain concepts.

In [340]:

def print_domain_model_summary(G: nx.DiGraph):
  """
  Print a simple summary of the domain model graph.
  """
  print("Domain Model Summary")
  print("--------------------")
  print(f"Classes: {G.number_of_nodes()}")
  print(f"Relationships: {G.number_of_edges()}")

  print("\nMost referenced domain classes:")
  for class_name, degree in sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {class_name}: referenced by {degree} classes")

  print("\nDomain classes referencing most others:")
  for class_name, degree in sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {class_name}: references {degree} classes")


print_domain_model_summary(DMG)


Domain Model Summary
--------------------
Classes: 113
Relationships: 314

Most referenced domain classes:
  User: referenced by 37 classes
  Article: referenced by 27 classes
  Language: referenced by 27 classes
  Bookmark: referenced by 12 classes
  Url: referenced by 11 classes
  UserWord: referenced by 11 classes
  ContextType: referenced by 7 classes
  Text: referenced by 7 classes
  UserCohortMap: referenced by 6 classes
  ContextIdentifier: referenced by 5 classes

Domain classes referencing most others:
  UnsignedBigInteger: references 22 classes
  Article: references 22 classes
  User: references 19 classes
  Bookmark: references 17 classes
  UserArticle: references 13 classes
  Video: references 12 classes
  UserWord: references 7 classes
  ContextIdentifier: references 6 classes
  ContextType: references 6 classes
  Meaning: references 5 classes


### 2.8 Show relationships as a table

This is an alternative to the graph, instead displaying the relationships in a table rather than in the graph

In [341]:
import pandas as pd

def domain_model_edges_as_dataframe(G: nx.DiGraph):
  """
  Convert domain model relationships into a pandas DataFrame.
  """

  rows = []

  for source, target in G.edges:
    rows.append({
      "Source class": source,
      "Target class": target,
      "Interpretation": f"{source} references {target}"
    })

  return pd.DataFrame(rows)


domain_relationships_df = domain_model_edges_as_dataframe(DMG)
domain_relationships_df.head(DMG.number_of_edges())


,Source class,Target class,Interpretation
0,AIGenerator,Text,AIGenerator references Text
1,UnsignedBigInteger,Article,UnsignedBigInteger references Article
2,UnsignedBigInteger,ArticleUrlKeywordMap,UnsignedBigInteger references ArticleUrlKeywor...
3,UnsignedBigInteger,UserArticle,UnsignedBigInteger references UserArticle
4,UnsignedBigInteger,ContextType,UnsignedBigInteger references ContextType
...,...,...,...
309,VideoCaptionContext,Bookmark,VideoCaptionContext references Bookmark
310,VideoTitleContext,UserWord,VideoTitleContext references UserWord
311,VideoTitleContext,Bookmark,VideoTitleContext references Bookmark
312,YTChannel,Language,YTChannel references Language


### 2.9 Relationships to specific classes

There's a lot of relationships in the Domain model, and looking at all of them at once, might not necessarily be too helpful. 
Instead having methods singling in on specific classes, such as the most common ones could be helpful.

In [342]:
def classes_targeting_model_class_df(G: nx.DiGraph, target_class: str):
  """
  Return a pandas DataFrame showing which model classes target/reference
  a specific model class.

  In the domain model graph:
    A -> B means A references B

  Therefore, incoming edges to target_class show which classes reference it.
  """

  if target_class not in G.nodes:
    available_classes = sorted(G.nodes)

    return pd.DataFrame({
      "error": [f"Class '{target_class}' was not found in the domain model graph."],
      "available_classes": [", ".join(available_classes)]
    })

  rows = []

  for source, target in G.in_edges(target_class):
    rows.append({
      "source_class": source,
      "target_class": target,
      "direction": "incoming",
      "interpretation": f"{source} references {target}"
    })

  return pd.DataFrame(
    rows,
    columns=[
      "source_class",
      "target_class",
      "direction",
      "interpretation"
    ]
  )

def classes_targeted_by_model_class_df(G: nx.DiGraph, source_class: str):
  """
  Return a pandas DataFrame showing which model classes are targeted/referenced
  by a specific model class.

  In the domain model graph:
    A -> B means A references B

  Therefore, outgoing edges from source_class show which classes it references.
  """

  if source_class not in G.nodes:
    available_classes = sorted(G.nodes)

    return pd.DataFrame({
      "error": [f"Class '{source_class}' was not found in the domain model graph."],
      "available_classes": [", ".join(available_classes)]
    })

  rows = []

  for source, target in G.out_edges(source_class):
    rows.append({
      "source_class": source,
      "target_class": target,
      "direction": "outgoing",
      "interpretation": f"{source} references {target}"
    })

  return pd.DataFrame(
    rows,
    columns=[
      "source_class",
      "target_class",
      "direction",
      "interpretation"
    ]
  )

### 2.10 Show relationships of most common domain classes: User, Article, Bookmark

In [343]:
# User
classes_targeting_user_df = classes_targeting_model_class_df(DMG, "User")
classes_targeted_by_user_df = classes_targeted_by_model_class_df(DMG, "User")

# Bookmark
classes_targeting_bookmark_df = classes_targeting_model_class_df(DMG, "Bookmark")
classes_targeted_by_bookmark_df = classes_targeted_by_model_class_df(DMG, "Bookmark")

# Article
classes_targeting_article_df = classes_targeting_model_class_df(DMG, "Article")
classes_targeted_by_article_df = classes_targeted_by_model_class_df(DMG, "Article")

# Language
classes_targeting_language_df = classes_targeting_model_class_df(DMG, "Language")

In [344]:
classes_targeting_user_df

,source_class,target_class,direction,interpretation
0,UnsignedBigInteger,User,incoming,UnsignedBigInteger references User
1,Article,User,incoming,Article references User
2,ArticleDifficultyFeedback,User,incoming,ArticleDifficultyFeedback references User
3,ArticleTopicUserFeedback,User,incoming,ArticleTopicUserFeedback references User
4,ArticleUpload,User,incoming,ArticleUpload references User
5,AudioLessonGenerationProgress,User,incoming,AudioLessonGenerationProgress references User
6,Cohort,User,incoming,Cohort references User
7,DailyAudioLesson,User,incoming,DailyAudioLesson references User
8,ExampleSentence,User,incoming,ExampleSentence references User
9,Friendship,User,incoming,Friendship references User


In [345]:
classes_targeted_by_user_df

,source_class,target_class,direction,interpretation
0,User,Exercise,outgoing,User references Exercise
1,User,UserPreference,outgoing,User references UserPreference
2,User,Cohort,outgoing,User references Cohort
3,User,UserAvatar,outgoing,User references UserAvatar
4,User,Article,outgoing,User references Article
5,User,UserLanguage,outgoing,User references UserLanguage
6,User,Teacher,outgoing,User references Teacher
7,User,UserArticle,outgoing,User references UserArticle
8,User,Meaning,outgoing,User references Meaning
9,User,CohortArticleMap,outgoing,User references CohortArticleMap


In [346]:
classes_targeting_bookmark_df

,source_class,target_class,direction,interpretation
0,ArticleFragmentContext,Bookmark,incoming,ArticleFragmentContext references Bookmark
1,ArticleSummaryContext,Bookmark,incoming,ArticleSummaryContext references Bookmark
2,ArticleTitleContext,Bookmark,incoming,ArticleTitleContext references Bookmark
3,BookmarkContext,Bookmark,incoming,BookmarkContext references Bookmark
4,ExampleSentenceContext,Bookmark,incoming,ExampleSentenceContext references Bookmark
5,Text,Bookmark,incoming,Text references Bookmark
6,User,Bookmark,incoming,User references Bookmark
7,UserArticle,Bookmark,incoming,UserArticle references Bookmark
8,UserVideo,Bookmark,incoming,UserVideo references Bookmark
9,UserWord,Bookmark,incoming,UserWord references Bookmark


In [347]:
classes_targeted_by_bookmark_df

,source_class,target_class,direction,interpretation
0,Bookmark,ContextType,outgoing,Bookmark references ContextType
1,Bookmark,ContextIdentifier,outgoing,Bookmark references ContextIdentifier
2,Bookmark,Meaning,outgoing,Bookmark references Meaning
3,Bookmark,ArticleFragmentContext,outgoing,Bookmark references ArticleFragmentContext
4,Bookmark,ArticleTitleContext,outgoing,Bookmark references ArticleTitleContext
5,Bookmark,Source,outgoing,Bookmark references Source
6,Bookmark,BookmarkContext,outgoing,Bookmark references BookmarkContext
7,Bookmark,Article,outgoing,Bookmark references Article
8,Bookmark,UserWord,outgoing,Bookmark references UserWord
9,Bookmark,UserWordInteractionHistory,outgoing,Bookmark references UserWordInteractionHistory


In [348]:
classes_targeting_article_df

,source_class,target_class,direction,interpretation
0,UnsignedBigInteger,Article,incoming,UnsignedBigInteger references Article
1,LowQualityTypes,Article,incoming,LowQualityTypes references Article
2,ArticleBrokenMap,Article,incoming,ArticleBrokenMap references Article
3,ClassificationType,Article,incoming,ClassificationType references Article
4,DetectionMethod,Article,incoming,DetectionMethod references Article
5,ArticleClassification,Article,incoming,ArticleClassification references Article
6,ArticleDifficultyFeedback,Article,incoming,ArticleDifficultyFeedback references Article
7,ArticleFragment,Article,incoming,ArticleFragment references Article
8,ArticleSummaryContext,Article,incoming,ArticleSummaryContext references Article
9,ArticleTitleContext,Article,incoming,ArticleTitleContext references Article


In [349]:
classes_targeted_by_article_df

,source_class,target_class,direction,interpretation
0,Article,ArticleUrlKeywordMap,outgoing,Article references ArticleUrlKeywordMap
1,Article,UserArticle,outgoing,Article references UserArticle
2,Article,ContextType,outgoing,Article references ContextType
3,Article,ArticleTokenizationCache,outgoing,Article references ArticleTokenizationCache
4,Article,ArticleCefrAssessment,outgoing,Article references ArticleCefrAssessment
5,Article,CohortArticleMap,outgoing,Article references CohortArticleMap
6,Article,ArticleFragment,outgoing,Article references ArticleFragment
7,Article,User,outgoing,Article references User
8,Article,Language,outgoing,Article references Language
9,Article,Feed,outgoing,Article references Feed


In [350]:
classes_targeting_language_df

,source_class,target_class,direction,interpretation
0,UnsignedBigInteger,Language,incoming,UnsignedBigInteger references Language
1,Article,Language,incoming,Article references Language
2,ArticleUpload,Language,incoming,ArticleUpload references Language
3,AudioLessonDialogue,Language,incoming,AudioLessonDialogue references Language
4,AudioLessonMeaning,Language,incoming,AudioLessonMeaning references Language
5,BookmarkContext,Language,incoming,BookmarkContext references Language
6,Cohort,Language,incoming,Cohort references Language
7,DailyAudioLesson,Language,incoming,DailyAudioLesson references Language
8,ExampleSentence,Language,incoming,ExampleSentence references Language
9,Feed,Language,incoming,Feed references Language


## 3. Improving the Domain Model View

The current Domain Model View only shows that the domain classes are connected, not how they are connected. 
In the following sections I will try to make new methods that attempt to show how the domain classes are connected. Adding meaning to the connections by classifying the relationship types and improving visualization with colors, labels, and node sizes.

### 3.1 Extract model class definitions

Unlike before this function does not just store the class name. It also stores:
- The module where the class was found
- the file path
- the actual AST node for the class

Keeping the AST node is important, because we later will inspect the inside of each class to find relationships.

In [351]:

def model_class_definitions(code_root_folder: str, model_package: str = "zeeguu.core.model") -> dict:
    """
    Find class definitions inside the model package.

    Returns:
      {
        "User": {
          "module": "zeeguu.core.model.user",
          "file": "../zeeguu-api/zeeguu/core/model/user.py",
          "node": ast.ClassDef(...)
        }
      }
    """
    classes = {}

    for file in Path(code_root_folder).rglob("*.py"):
      file = str(file)
      module_name = module_name_from_file_path(file)

      if not module_name.startswith(model_package):
        continue

      with open(file, encoding="utf-8", errors="ignore") as f:
        tree = ast.parse(f.read(), filename=file)

      for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
          classes[node.name] = {
            "module": module_name,
            "file": file,
            "node": node
          }

    return classes


### 3.2 Helper functions for AST analysis

The Python ast module represents code as syntax-tree objects rather than normal strings. For example, a class name, a function call, or an attribute access are represented as different AST node types.
These helper functions convert common AST nodes into readable names.

In [380]:

def name_from_ast_node(node):
  """
  Convert common AST node types into readable names.

  Examples:
    ast.Name(id="User") -> "User"
    ast.Attribute(value=Name("db"), attr="Model") -> "db.Model"
  """
  if isinstance(node, ast.Name):
    return node.id

  if isinstance(node, ast.Attribute):
    parent = name_from_ast_node(node.value)
    if parent:
      return f"{parent}.{node.attr}"
    return node.attr

  if isinstance(node, ast.Constant):
    return str(node.value)

  if isinstance(node, ast.Str):  # Older Python compatibility
    return node.s

  return None


def call_name(node):
  """
  Get the function name from a Call node.

  Examples:
    relationship(...)
    db.relationship(...)
    ForeignKey(...)
    db.ForeignKey(...)
  """
  if isinstance(node, ast.Call):
    return name_from_ast_node(node.func)
  return None

def foreign_key_target_from_value(fk_value: str, known_classes: set[str]):
  """
  Infer the target model class from a ForeignKey value.

  Handles examples like:
    "language.id"
    "Language.id"
    "user.id"
    "User.id"
  """
  if not fk_value:
    return None

  fk_value_lower = fk_value.lower()

  for known_class in known_classes:
    class_name_lower = known_class.lower()

    # Handles: Language.id, language.id, User.id, user.id
    if fk_value_lower.startswith(class_name_lower + "."):
      return known_class

  return None

### 3.3 Extract Relationships from each Class

This is the main analysis function of the improved domain model view.
For each class, it inspects the class body and tries to find relationships to other known model classes.

It detects 4 types of relationships:
1. Inheritance
2. SQLAlchemy relationships
3. Foreign keys
4. General references

Inheritance is relationships are included if the model includes both the Parent class and the Child class. This type of relationship means that "A is a B". 
Notably I choose to ignore inheritance from other classes, that aren't part of the model, as those aren't domain inheritance. They will typically be inheritance from database base classes.

SQLAlchemy relationships are relationships that occur when one Domain object has an association to another. This is ORM/object-level relationship.

Foreign keys means that there is a database foreign key reference to another class. This means there a database-level dependency between the two domain classes.

General references mean that one model class mentions another known model class somewhere in the code. There are many ways this could happen, and doesn't necessarily mean theres a domain relationship.


#### Strength of each relationship type
Inheritance, SQLAlchemy relationships, and foreign keys are stronger relationship types because they are explicit structural signals in the code. 
They usually indicate that two classes are connected at the domain or persistence level.

General references is weaker because it only means that one class mentions or uses another class somewhere in its implementation.




In [381]:
def extract_relationships_from_class(class_name: str, class_node: ast.ClassDef, known_classes: set[str]) -> list:
  """
  Extract domain-model relationships from a class AST node.

  Returns a list of relationships:
    {
      "source": "Bookmark",
      "target": "User",
      "type": "foreign_key",
      "detail": "ForeignKey('user.id')"
    }
  """
  relationships = []

  # 1. Inheritance relationships
  for base in class_node.bases:
    base_name = name_from_ast_node(base)

    if base_name in known_classes:
      relationships.append({
        "source": class_name,
        "target": base_name,
        "type": "inherits",
        "detail": f"{class_name} inherits from {base_name}"
      })

  # 2. Walk inside the class body
  for node in ast.walk(class_node):
    # General references to other model classes
    if isinstance(node, ast.Name):
      if node.id in known_classes and node.id != class_name:
        relationships.append({
          "source": class_name,
          "target": node.id,
          "type": "reference",
          "detail": f"{class_name} references {node.id}"
        })

    # Function calls: relationship(...), ForeignKey(...), etc.
    if isinstance(node, ast.Call):
      function_name = call_name(node)

      # SQLAlchemy relationship(...)
      if function_name and function_name.endswith("relationship"):
        for arg in node.args:
          target = name_from_ast_node(arg)

          if target in known_classes and target != class_name:
            relationships.append({
              "source": class_name,
              "target": target,
              "type": "relationship",
              "detail": f"{class_name} has relationship to {target}"
            })

      # SQLAlchemy ForeignKey(...)
      if function_name and function_name.endswith("ForeignKey"):
        for arg in node.args:
          fk_value = name_from_ast_node(arg)

          target_class = foreign_key_target_from_value(
            fk_value,
            known_classes
          )

          if target_class and target_class != class_name:
            relationships.append({
              "source": class_name,
              "target": target_class,
              "type": "foreign_key",
              "detail": f"{class_name} has foreign key to {target_class}: {fk_value}"
            })


  return relationships


### 3.4 Build the Improved Domain Model Graph

Now the improved Domain Model Graph can be built.
This function turns the extracted classes and relationships into a directed graph. 

Like with the previous graphs.
Each node in the graph is a model class.
Each edge is a relationship from one model class to another.

As a default general references are excluded, since they will often add noise without being relevant to the actual domain model.

In [382]:
def improved_domain_model_digraph(
  code_root_folder: str,
  model_package: str = "zeeguu.core.model",
  relationship_types_to_include: set[str] | None = None,
  keep_isolated_nodes: bool = True
) -> nx.DiGraph:
  """
  Build an improved domain model graph.

  Nodes are model classes.
  Edges are classified as:
    - inherits
    - relationship
    - foreign_key
    - reference

  Parameters:
    code_root_folder:
      Path to the root folder of the codebase.

    model_package:
      The model package to analyse.

    relationship_types_to_include:
      Optional set of relationship types to include.

      Example:
        {"foreign_key"}
        {"relationship", "foreign_key"}
        {"inherits", "relationship", "foreign_key"}
        {"inherits", "relationship", "foreign_key", "reference"}

      If None, the default is:
        {"inherits", "relationship", "foreign_key"}

      This means general references are excluded by default because they
      can make the graph noisy.

    keep_isolated_nodes:
      If True, all model classes are added as nodes, even if they do not
      have any included relationships.
      If False, only classes participating in included relationships are shown.
  """

  if relationship_types_to_include is None:
    relationship_types_to_include = {
      "inherits",
      "relationship",
      "foreign_key"
    }

  class_defs = model_class_definitions(code_root_folder, model_package)
  known_classes = set(class_defs.keys())

  G = nx.DiGraph()

  # Add all nodes first, unless we only want nodes with included relationships.
  if keep_isolated_nodes:
    for class_name, data in class_defs.items():
      G.add_node(
        class_name,
        module=data["module"],
        file=data["file"]
      )

  # Add edges
  for class_name, data in class_defs.items():
    relationships = extract_relationships_from_class(
      class_name,
      data["node"],
      known_classes
    )

    for rel in relationships:
      rel_type = rel["type"]

      # Only include the selected relationship types
      if rel_type not in relationship_types_to_include:
        continue

      source = rel["source"]
      target = rel["target"]
      detail = rel["detail"]

      if source == target:
        continue

      # Add nodes only when needed if keep_isolated_nodes=False
      if not G.has_node(source):
        G.add_node(
          source,
          module=class_defs[source]["module"],
          file=class_defs[source]["file"]
        )

      if not G.has_node(target):
        G.add_node(
          target,
          module=class_defs[target]["module"],
          file=class_defs[target]["file"]
        )

      # If an edge already exists, collect multiple relationship types
      if G.has_edge(source, target):
        existing_types = G[source][target].get("types", set())
        existing_details = G[source][target].get("details", [])

        existing_types.add(rel_type)
        existing_details.append(detail)

        G[source][target]["types"] = existing_types
        G[source][target]["details"] = existing_details
      else:
        G.add_edge(
          source,
          target,
          types={rel_type},
          details=[detail]
        )

  return G


### 3.5 Draw the Improved Domain Model Graph

In [383]:
def draw_improved_domain_model_graph(
  G: nx.DiGraph,
  figure_name: str,
  height: str = "900px",
  width: str = "100%",
  notebook: bool = True
):
  """
  Draw an improved domain model graph.

  Node size is based on total degree.
  Edge color is based on relationship type.
  """
  os.makedirs("./graphs", exist_ok=True)

  net = Network(
    height=height,
    width=width,
    directed=True,
    notebook=notebook,
    bgcolor="#ffffff",
    font_color="#000000",
  )

  net.barnes_hut(
    gravity=-30000,
    central_gravity=0.3,
    spring_length=220,
    spring_strength=0.04,
    damping=0.09,
    overlap=0,
  )

  edge_colors = {
    "inherits": "#8e44ad",      # purple
    "relationship": "#27ae60",  # green
    "foreign_key": "#e67e22",   # orange
    "reference": "#95a5a6",     # grey
  }

  edge_widths = {
    "inherits": 4,
    "relationship": 3,
    "foreign_key": 3,
    "reference": 1,
  }

  # Add nodes with size based on degree
  for node, data in G.nodes(data=True):
    degree = G.in_degree(node) + G.out_degree(node)
    size = 15 + degree * 4

    module_name = data.get("module", "")
    file_name = data.get("file", "")

    title = f"""
    <b>{node}</b><br>
    Module: {module_name}<br>
    File: {file_name}<br>
    Incoming relationships: {G.in_degree(node)}<br>
    Outgoing relationships: {G.out_degree(node)}
    """

    net.add_node(
      node,
      label=node,
      title=title,
      size=size,
    )

  # Add edges
  for source, target, data in G.edges(data=True):
    relationship_types = data.get("types", set())
    details = data.get("details", [])

    # Pick strongest relationship type for visual styling
    priority = ["inherits", "relationship", "foreign_key", "reference"]
    main_type = next(
      rel_type for rel_type in priority
      if rel_type in relationship_types
    )

    color = edge_colors.get(main_type, "#95a5a6")
    width = edge_widths.get(main_type, 1)

    title = "<br>".join(details)

    net.add_edge(
      source,
      target,
      title=title,
      label=main_type,
      color=color,
      width=width,
    )

  net.show_buttons(filter_=["physics"])

  output_path = f"./graphs/{figure_name}.html"
  net.write_html(output_path)

  print(f"Saved improved domain model graph to: {output_path}")
  #display(IFrame(src=output_path, width="100%", height=height.replace("px", "")))

  return output_path


### 3.6 Generate the Improved Domain Model View

In [384]:

IDMG = improved_domain_model_digraph(
  CODE_ROOT_FOLDER,
  relationship_types_to_include={
    "inherits", 
    "foreign_key", 
    "relationship"
  }
)


print(
  f"Improved domain model graph: "
  f"{IDMG.number_of_nodes()} classes, "
  f"{IDMG.number_of_edges()} relationships"
)

draw_improved_domain_model_graph(IDMG, "improved-domain-model-view")


Improved domain model graph: 113 classes, 177 relationships
Saved improved domain model graph to: ./graphs/improved-domain-model-view.html


/tmp/ipykernel_7992/4109086330.py:21: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Str):  # Older Python compatibility


'./graphs/improved-domain-model-view.html'

In [385]:
IDMG_with_refs = improved_domain_model_digraph(
  CODE_ROOT_FOLDER,
  relationship_types_to_include={
    "inherits",
    "relationship",
    "foreign_key",
    "reference"
  }
)


print(
  f"Improved domain model graph with references: "
  f"{IDMG_with_refs.number_of_nodes()} classes, "
  f"{IDMG_with_refs.number_of_edges()} relationships"
)

draw_improved_domain_model_graph(
  IDMG_with_refs,
  "improved-domain-model-view-with-references"
)


Improved domain model graph with references: 113 classes, 310 relationships
Saved improved domain model graph to: ./graphs/improved-domain-model-view-with-references.html


/tmp/ipykernel_7992/4109086330.py:21: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Str):  # Older Python compatibility


'./graphs/improved-domain-model-view-with-references.html'

### 3.7 Metadata on graph

Now that we have graphs, I check how many of each type of relationship there is in the graph.

In [386]:
from collections import Counter

def relationship_type_combinations_as_dataframe(G: nx.DiGraph):
  """
  Count combinations of relationship types on graph edges.

  This shows how often edges have multiple labels, e.g.
  reference + relationship, or reference + foreign_key.
  """

  counter = Counter()

  for source, target, data in G.edges(data=True):
    relationship_types = data.get("types", set())

    if relationship_types:
      combination = " + ".join(sorted(relationship_types))
    else:
      combination = "unknown"

    counter[combination] += 1

  rows = []

  for combination, count in counter.items():
    rows.append({
      "Relationship type combination": combination,
      "Edge count": count
    })

  df = pd.DataFrame(rows)

  if not df.empty:
    df = df.sort_values("Edge count", ascending=False).reset_index(drop=True)

  return df



In [387]:
relationship_type_counts_df = relationship_type_combinations_as_dataframe(IDMG_with_refs)
relationship_type_counts_df

,Relationship type combination,Edge count
0,reference,133
1,foreign_key + reference + relationship,129
2,foreign_key + relationship,22
3,reference + relationship,13
4,relationship,11
5,foreign_key + reference,2


### 3.8 Only Foreign Keys

Since in the previous graphs no foreign keys exists without also having another type of relationship, I make a graph of only foreign_key relationships.

In [388]:
FK_DMG = improved_domain_model_digraph(
  CODE_ROOT_FOLDER,
  relationship_types_to_include={
    "foreign_key"
  },
  keep_isolated_nodes=False
)

print(
  f"Improved domain model graph with only foreign keys: "
  f"{FK_DMG.number_of_nodes()} classes, "
  f"{FK_DMG.number_of_edges()} relationships"
)

draw_improved_domain_model_graph(
  FK_DMG,
  "improved-domain-model-view-only-foreign-keys"
)

Improved domain model graph with only foreign keys: 93 classes, 153 relationships
Saved improved domain model graph to: ./graphs/improved-domain-model-view-only-foreign-keys.html


/tmp/ipykernel_7992/4109086330.py:21: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Str):  # Older Python compatibility


'./graphs/improved-domain-model-view-only-foreign-keys.html'